In [1]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [2]:
# image load => transform => dataset of all imgs
class ImageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of paths of all images
        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)
    
    def __getitem__(self, idx):
         img_path = self.all_img_paths[idx]
         img = Image.open(img_path).convert("RGB")

         if self.transformations:
             img = self.transformations(img)
 
         return img

In [3]:
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178 x 218 => 178 x 178
    transforms.Resize(64), # 64 x 64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # [-1, 1]

])

In [4]:
root_dir_path = "./img_align_celeba"

In [5]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 67715 images


In [6]:
dataloader = DataLoader(dataset, batch_size=128, shuffle = True)

In [7]:
os.listdir(root_dir_path)

['000001.jpg',
 '000002.jpg',
 '000003.jpg',
 '000004.jpg',
 '000005.jpg',
 '000006.jpg',
 '000007.jpg',
 '000008.jpg',
 '000009.jpg',
 '000010.jpg',
 '000011.jpg',
 '000012.jpg',
 '000013.jpg',
 '000014.jpg',
 '000015.jpg',
 '000016.jpg',
 '000017.jpg',
 '000018.jpg',
 '000019.jpg',
 '000020.jpg',
 '000021.jpg',
 '000022.jpg',
 '000023.jpg',
 '000024.jpg',
 '000025.jpg',
 '000026.jpg',
 '000027.jpg',
 '000028.jpg',
 '000029.jpg',
 '000030.jpg',
 '000031.jpg',
 '000032.jpg',
 '000033.jpg',
 '000034.jpg',
 '000035.jpg',
 '000036.jpg',
 '000037.jpg',
 '000038.jpg',
 '000039.jpg',
 '000040.jpg',
 '000041.jpg',
 '000042.jpg',
 '000043.jpg',
 '000044.jpg',
 '000045.jpg',
 '000046.jpg',
 '000047.jpg',
 '000048.jpg',
 '000049.jpg',
 '000050.jpg',
 '000051.jpg',
 '000052.jpg',
 '000053.jpg',
 '000054.jpg',
 '000055.jpg',
 '000056.jpg',
 '000057.jpg',
 '000058.jpg',
 '000059.jpg',
 '000060.jpg',
 '000061.jpg',
 '000062.jpg',
 '000063.jpg',
 '000064.jpg',
 '000065.jpg',
 '000066.jpg',
 '000067.j

In [8]:
for img in os.listdir(root_dir_path):
    print(os.path.join(root_dir_path, img))

./img_align_celeba\000001.jpg
./img_align_celeba\000002.jpg
./img_align_celeba\000003.jpg
./img_align_celeba\000004.jpg
./img_align_celeba\000005.jpg
./img_align_celeba\000006.jpg
./img_align_celeba\000007.jpg
./img_align_celeba\000008.jpg
./img_align_celeba\000009.jpg
./img_align_celeba\000010.jpg
./img_align_celeba\000011.jpg
./img_align_celeba\000012.jpg
./img_align_celeba\000013.jpg
./img_align_celeba\000014.jpg
./img_align_celeba\000015.jpg
./img_align_celeba\000016.jpg
./img_align_celeba\000017.jpg
./img_align_celeba\000018.jpg
./img_align_celeba\000019.jpg
./img_align_celeba\000020.jpg
./img_align_celeba\000021.jpg
./img_align_celeba\000022.jpg
./img_align_celeba\000023.jpg
./img_align_celeba\000024.jpg
./img_align_celeba\000025.jpg
./img_align_celeba\000026.jpg
./img_align_celeba\000027.jpg
./img_align_celeba\000028.jpg
./img_align_celeba\000029.jpg
./img_align_celeba\000030.jpg
./img_align_celeba\000031.jpg
./img_align_celeba\000032.jpg
./img_align_celeba\000033.jpg
./img_alig

## Generator Network

In [9]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels=3): # 3 is for rgb 
        super(Generator, self).__init__()

        # fully connected (dense) layer
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 -> 256
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64 * 64 * img_channels),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img
    
        # fake img => 64 x 64 x 3 x batch_size => 4d tensor

## Discriminator Network

In [12]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3): # 3 is for rgb 
        super(Discriminator, self).__init__()

        # fully connected (dense) layer
        self.model = nn.Sequential(
            nn.Flatten(),  # 4D tensor -> 1D

            nn.Linear(img_channels * 64 * 64, 1024), # 100 -> 256
            nn.LeakyReLU(0.2, inplace = True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),
            nn.LeakyReLU(),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.model(img)

In [13]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [14]:
import torch
# device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device is {device}")

Device is cuda


In [15]:
generator = generator.to(device)
discriminator = discriminator.to(device)

## Training the GAN

In [20]:
def train(generator, discriminator, dataloader, epochs=10):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # Create real imgs labels & fake imgs labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # Train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # Train the generator
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels) #(y_pred, y_actual)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch {epoch + 1}/{epochs}... batch: {i + 1}... G-loss:{g_loss}... D-loss: {d_loss}")
        
        # Save generated imgae for each epoch
        save_generated_images(generator, epoch, device)



In [23]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    # [-1, 1] => [0, 1]
    grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [24]:
train(generator, discriminator, dataloader, epochs=2)

for epoch 1/2... batch: 1... G-loss:0.9192174077033997... D-loss: 0.29281044006347656
for epoch 1/2... batch: 51... G-loss:0.9676392078399658... D-loss: 0.42443588376045227
for epoch 1/2... batch: 101... G-loss:0.9423071146011353... D-loss: 0.32677462697029114
for epoch 1/2... batch: 151... G-loss:1.2690080404281616... D-loss: 0.2842863202095032
for epoch 1/2... batch: 201... G-loss:1.3204032182693481... D-loss: 0.16526547074317932
for epoch 1/2... batch: 251... G-loss:2.2551891803741455... D-loss: 0.22666913270950317
for epoch 1/2... batch: 301... G-loss:0.6699580550193787... D-loss: 0.6430251002311707
for epoch 1/2... batch: 351... G-loss:1.4597889184951782... D-loss: 0.315365731716156
for epoch 1/2... batch: 401... G-loss:2.809615135192871... D-loss: 0.2753717005252838
for epoch 1/2... batch: 451... G-loss:2.3644440174102783... D-loss: 0.1312883496284485
for epoch 1/2... batch: 501... G-loss:2.9302446842193604... D-loss: 0.21443669497966766


: 